# Credit Risk Analytics & Loan Default Prediction

## Stage 3 — Data Preparation

This notebook prepares the credit-card default dataset for machine-learning modelling.

The main objectives are to:

- load the original dataset
- validate the data and target variable
- clean problematic categorical values
- engineer useful credit-risk features
- select appropriate modelling variables
- separate features from the target
- create training and testing datasets

The prepared data will later be used to develop and evaluate interpretable credit-risk classification models.


In [15]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [16]:
# Load Default of Credit Card Clients dataset

credit_data = fetch_ucirepo(id=350)

X = credit_data.data.features.copy()
y = credit_data.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (30000, 23)
Target shape: (30000, 1)


In [17]:
# Rename columns to meaningful names

column_names = {
    "X1": "LIMIT_BAL",
    "X2": "SEX",
    "X3": "EDUCATION",
    "X4": "MARRIAGE",
    "X5": "AGE",
    "X6": "PAY_0",
    "X7": "PAY_2",
    "X8": "PAY_3",
    "X9": "PAY_4",
    "X10": "PAY_5",
    "X11": "PAY_6",
    "X12": "BILL_AMT1",
    "X13": "BILL_AMT2",
    "X14": "BILL_AMT3",
    "X15": "BILL_AMT4",
    "X16": "BILL_AMT5",
    "X17": "BILL_AMT6",
    "X18": "PAY_AMT1",
    "X19": "PAY_AMT2",
    "X20": "PAY_AMT3",
    "X21": "PAY_AMT4",
    "X22": "PAY_AMT5",
    "X23": "PAY_AMT6"
}

X = X.rename(columns=column_names)
y = y.rename(columns={"Y": "DEFAULT"})

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.columns.tolist())

Feature columns:
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

Target column:
['DEFAULT']


In [18]:
# Combine features and target into one DataFrame

df = X.copy()
df["DEFAULT"] = y["DEFAULT"]

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [19]:
# Check for missing values

print("Total missing values:", df.isnull().sum().sum())

print("\nMissing values by column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Total missing values: 0

Missing values by column:
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_0        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
DEFAULT      0
dtype: int64

Duplicate rows: 35


In [20]:
# Inspect categorical variables for unusual values

for column in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(f"\n{column}")
    print(df[column].value_counts().sort_index())


SEX
SEX
1    11888
2    18112
Name: count, dtype: int64

EDUCATION
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

MARRIAGE
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64


## Data Cleaning

Initial validation identified no missing values but found 35 duplicate records.

The EDUCATION and MARRIAGE variables also contain undocumented or uncommon category codes. These categories will be consolidated into an "Other" category rather than removing the observations, preserving the available customer information while simplifying the variables for modelling.


In [21]:
# Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))
print("Remaining duplicates:", df.duplicated().sum())


Rows after removing duplicates: 29965
Remaining duplicates: 0


In [22]:
# Consolidate uncommon/undocumented categories

# EDUCATION:
# 1 = Graduate school
# 2 = University
# 3 = High school
# 4 = Other
# 0, 5 and 6 are consolidated into Other
df["EDUCATION"] = df["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})

# MARRIAGE:
# 1 = Married
# 2 = Single
# 3 = Other
# 0 is consolidated into Other
df["MARRIAGE"] = df["MARRIAGE"].replace({
    0: 3
})

print("EDUCATION:")
print(df["EDUCATION"].value_counts().sort_index())

print("\nMARRIAGE:")
print(df["MARRIAGE"].value_counts().sort_index())

EDUCATION:
EDUCATION
1    10563
2    14019
3     4915
4      468
Name: count, dtype: int64

MARRIAGE:
MARRIAGE
1    13643
2    15945
3      377
Name: count, dtype: int64


## Feature Engineering

Additional behavioural features are created from the six months of historical account data. These variables summarise repayment delays, bill amounts, payments and credit utilisation to provide interpretable indicators of customer credit risk.# Historical variable groups

repayment_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]

payment_cols = [
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

In [24]:
# Historical variable groups

repayment_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]

payment_cols = [
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

In [25]:
# Engineer behavioural risk features

df["NUM_DELAYED_MONTHS"] = (df[repayment_cols] > 0).sum(axis=1)

df["MAX_DELAY"] = df[repayment_cols].max(axis=1)

df["AVG_BILL_AMT"] = df[bill_cols].mean(axis=1)

df["AVG_PAY_AMT"] = df[payment_cols].mean(axis=1)

df["UTILISATION"] = (
    df["BILL_AMT1"] / df["LIMIT_BAL"]
).clip(lower=0)

df["PAYMENT_TO_BILL"] = (
    df["AVG_PAY_AMT"] /
    df["AVG_BILL_AMT"].replace(0, float("nan"))
)

print("Feature engineering complete.")
print("Dataset shape:", df.shape)

Feature engineering complete.
Dataset shape: (29965, 30)


In [26]:
engineered_features = [
    "NUM_DELAYED_MONTHS",
    "MAX_DELAY",
    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "UTILISATION",
    "PAYMENT_TO_BILL"
]

df[engineered_features].describe().round(2)

,NUM_DELAYED_MONTHS,MAX_DELAY,AVG_BILL_AMT,AVG_PAY_AMT,UTILISATION,PAYMENT_TO_BILL
count,29965.00,29965.00,29965.00,29965.00,29965.00,29125.00
mean,0.83,0.44,45029.33,5281.30,0.42,0.39
std,1.55,1.34,63279.07,10142.31,0.41,7.79
min,0.00,-2.00,-56043.17,0.00,0.00,-546.93
25%,0.00,0.00,4841.00,1116.67,0.02,0.04
50%,0.00,0.00,21110.83,2400.00,0.32,0.09
75%,1.00,2.00,57166.17,5600.00,0.83,0.62
max,6.00,8.00,877313.83,627344.33,6.46,797.00


In [27]:
import numpy as np

print("Missing values:")
print(df[engineered_features].isnull().sum())

print("\nInfinite values:")
print(
    np.isinf(
        df[engineered_features].select_dtypes(include=np.number)
    ).sum()
)

Missing values:
NUM_DELAYED_MONTHS      0
MAX_DELAY               0
AVG_BILL_AMT            0
AVG_PAY_AMT             0
UTILISATION             0
PAYMENT_TO_BILL       840
dtype: int64

Infinite values:
NUM_DELAYED_MONTHS    0
MAX_DELAY             0
AVG_BILL_AMT          0
AVG_PAY_AMT           0
UTILISATION           0
PAYMENT_TO_BILL       0
dtype: int64


In [28]:
# Clean payment-to-bill ratio

# Missing ratios occur where average bill amount is zero
df["PAYMENT_TO_BILL"] = df["PAYMENT_TO_BILL"].fillna(0)

# Cap extreme values to make the feature more robust
lower_limit = df["PAYMENT_TO_BILL"].quantile(0.01)
upper_limit = df["PAYMENT_TO_BILL"].quantile(0.99)

df["PAYMENT_TO_BILL"] = df["PAYMENT_TO_BILL"].clip(
    lower=lower_limit,
    upper=upper_limit
)

print("Missing PAYMENT_TO_BILL:", df["PAYMENT_TO_BILL"].isnull().sum())
print("Minimum:", round(df["PAYMENT_TO_BILL"].min(), 2))
print("Maximum:", round(df["PAYMENT_TO_BILL"].max(), 2))

Missing PAYMENT_TO_BILL: 0
Minimum: -0.0
Maximum: 2.73


In [29]:
# Select variables for modelling

feature_columns = [
    "LIMIT_BAL",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "AGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6",
    "NUM_DELAYED_MONTHS",
    "MAX_DELAY",
    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "UTILISATION",
    "PAYMENT_TO_BILL"
]

X = df[feature_columns].copy()
y = df["DEFAULT"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isnull().sum().sum())

X shape: (29965, 29)
y shape: (29965,)
Missing values in X: 0


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining default rate:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting default rate:")
print(y_test.value_counts(normalize=True).round(4))

Training rows: 23972
Testing rows: 5993

Training default rate:
DEFAULT
0    0.7787
1    0.2213
Name: proportion, dtype: float64

Testing default rate:
DEFAULT
0    0.7787
1    0.2213
Name: proportion, dtype: float64


In [31]:
# Save prepared datasets for modelling

X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Prepared datasets saved successfully.")

Prepared datasets saved successfully.
